# NumPy Refresher Notebook (ML Coding Interviews)

**Goal:** a quick, practical refresher of NumPy patterns that show up in ML/data-coding interviews: shapes, broadcasting, indexing, vectorization, sorting/gather, linear algebra, and a few ML-flavored “building blocks” (softmax, distances, PCA-ish SVD, MRR).

**How to use**
- Skim the headings, run the cells, then modify the examples.
- Treat every shape comment as sacred law.

**References (high-signal)**
- NumPy docs: Quickstart, Broadcasting, and “Absolute basics”
- SciPy Lecture Notes (NumPy intro + advanced NumPy)
- Python Data Science Handbook (NumPy chapter, as notebooks)
- “NumPy 100 Exercises” repo / Kaggle adaptation

_Generated on 2026-02-02_

In [3]:
import numpy as np

print("NumPy:", np.__version__)
rng = np.random.default_rng(0)  # preferred modern RNG

NumPy: 1.26.4


## 1) Array creation & dtypes

Know these by reflex: `array`, `zeros/ones/full`, `arange/linspace`, `eye`, `diag`, `random.default_rng`.

In [6]:
a = np.array([1, 2, 3])
b = np.array([[1, 2], [3, 4]], dtype=np.float32)
print(a, a.dtype, a.shape)
print(b, b.dtype, b.shape)
print('\n')


z = np.zeros((2, 3))
o = np.ones((2, 3), dtype=np.int32)
f = np.full((2, 3), fill_value=7)
print(z)
print(o)
print(f)
print('\n')


ar = np.arange(0, 10, 2)      # [0,2,4,6,8]
ls = np.linspace(0, 1, 5)     # 5 points inclusive  [0.   0.25 0.5  0.75 1.  ]
print(ar)
print(ls)
print('\n')

I = np.eye(3)
d = np.diag([10, 20, 30])
print(I)
print(d)
print('\n')


rng = np.random.default_rng(0)  # preferred modern RNG
x = rng.normal(size=(3, 4))
u = rng.uniform(low=-1, high=1, size=(3, 4))
ints = rng.integers(low=0, high=10, size=(3, 4))
print(x[:2])
print(u[:2])
print(ints)

[1 2 3] int64 (3,)
[[1. 2.]
 [3. 4.]] float32 (2, 2)


[[0. 0. 0.]
 [0. 0. 0.]]
[[1 1 1]
 [1 1 1]]
[[7 7 7]
 [7 7 7]]


[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]


[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[[10  0  0]
 [ 0 20  0]
 [ 0  0 30]]


[[-0.43643525 -1.16980191  1.73936788 -0.49591073]
 [ 0.32896963 -0.25857255  1.58347288  1.32036099]]
[[-1.19245691e-01  9.09180987e-01 -2.08372625e-04 -1.49542750e-01]
 [ 2.40426904e-01  9.90193010e-01  8.97887350e-01 -7.99097214e-02]]
[[0 4 2 7]
 [7 7 9 9]
 [1 1 1 7]]


**dtype quick notes**
- Most ML interview code works fine with `float64`, but production often uses `float32` for speed/memory.
- Convert with `astype`.

In [7]:
y = np.array([1, 2, 3], dtype=np.int64)
y32 = y.astype(np.float32)
print(y.dtype, "->", y32.dtype)

int64 -> float32


## 2) Shapes, reshape, views vs copies

Interview gotcha: **reshape usually returns a view** (no copy) when possible.
Use `.copy()` when you need independent memory.

In [9]:
X = np.arange(12)            # shape (12,)
X2 = X.reshape(3, 4)         # view (usually)
print("X:", X.shape, "X2:", X2.shape) #X: [ 0  1  2  3  4  5  6  7  8  9 10 11]
print('\n')

# ravel returns a 1D view when possible; flatten returns a copy
r = X2.ravel()
fl = X2.flatten()
print("ravel shares memory?", np.shares_memory(X2, r))
print("flatten shares memory?", np.shares_memory(X2, fl))
print('\n')

# transpose is typically a view with changed strides
Xt = X2.T
print("Xt shape:", Xt.shape, "shares memory?", np.shares_memory(X2, Xt))
print('\n')

# copy explicitly
Xc = X2.copy()
print("copy shares memory?", np.shares_memory(X2, Xc))

X: [ 0  1  2  3  4  5  6  7  8  9 10 11] (12,) X2: (3, 4)


ravel shares memory? True
flatten shares memory? False


Xt shape: (4, 3) shares memory? True


copy shares memory? False


## 3) Indexing patterns: slicing, boolean masks, fancy indexing

- **Slicing** (:`:`) returns a view when possible.
- **Fancy indexing** (arrays of indices) returns a copy.

In [20]:
A = np.arange(20).reshape(4, 5) #reshape returns view
print("A:\n", A)

# slicing: view
sub = A[:3:2, 1::2]
print("sub:\n", sub)
print("shares memory?", np.shares_memory(A, sub))
print('\n')

# boolean mask
mask = (A % 3 == 0)
vals = A[mask]   # 1D array of selected elements (copy)
print("num multiples of 3:", vals)
print("mask shape", mask.shape)
print('\n')

# fancy indexing: pick specific rows/cols
rows = np.array([0, 2, 3])
cols = np.array([1, 4, 0])
picked = A[rows, cols]  # element-wise pair selection
print("picked:", picked)

# np.where: indices or elementwise selection
idx = np.where(A > 12)
print("where indices (first few):", list(zip(idx[0][:3], idx[1][:3])))

B = np.where(A > 12, 1, 0)  # elementwise ternary B is of A's shape
print("B:\n", B)

print('\n')
print(type(idx))
print(type(idx[0]))

A:
 [[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]
sub:
 [[ 1  3]
 [11 13]]
shares memory? True


num multiples of 3: [ 0  3  6  9 12 15 18]
mask shape (4, 5)


picked: [ 1 14 15]
where indices (first few): [(2, 3), (2, 4), (3, 0)]
B:
 [[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 1 1]
 [1 1 1 1 1]]


<class 'tuple'>
<class 'numpy.ndarray'>


## 4) Broadcasting (the superpower)

Rule of thumb: **align shapes from the right**. A dimension matches if equal or one of them is 1.

`keepdims=True` is your friend: it keeps reduction results broadcastable.

In [23]:
X = rng.normal(size=(4, 3))  # (n=4, d=3)
mu = X.mean(axis=0)          # (d,)
X_centered = X - mu          # broadcasts (4,3) - (3,) -> (4,3)
print("X:", X.shape, "mu:", mu.shape, "X_centered:", X_centered.shape)

mu2 = X.mean(axis=0, keepdims=True)  # (1, d)
X_centered2 = X - mu2                # (4,3) - (1,3) -> (4,3)
print("mu2:", mu2.shape, "X_centered2:", X_centered2.shape)

# Classic pairwise distances via broadcasting:
# X: (n,d), C: (k,d) -> (n,k,d) differences -> (n,k) squared distances
n, k, d = 5, 3, 2
X = rng.normal(size=(n, d))
C = rng.normal(size=(k, d))
d2 = np.sum((X[:, None, :] - C[None, :, :])**2, axis=2)
print("X:", X.shape, X[:, None, :].shape,  "C:", C.shape, C[None, :, :].shape, "d2:", d2.shape, )

X: (4, 3) mu: (3,) X_centered: (4, 3)
mu2: (1, 3) X_centered2: (4, 3)
X: (5, 2) (5, 1, 2) C: (3, 2) (1, 3, 2) d2: (5, 3)


## 5) Ufuncs, reductions, and axis discipline

Ufuncs = fast elementwise functions (`np.exp`, `np.maximum`, `np.add`, …).

Reductions collapse an axis: `sum/mean/max/min/std`. Know what `axis` means.

In [24]:
X = rng.normal(size=(3, 4))
print("X:\n", X)

# elementwise
Y = np.exp(X) / (1 + np.exp(X))  # sigmoid
print("sigmoid(Y) shape:", Y.shape)

# reductions
print("sum over all:", X.sum())
print("sum over rows (axis=1):", X.sum(axis=1).shape)
print("sum over cols (axis=0):", X.sum(axis=0).shape)

# keepdims helps broadcasting later
row_sum = X.sum(axis=1, keepdims=True)  # (3,1)
X_row_norm = X / row_sum                # broadcast (3,4)/(3,1)->(3,4)
print("row_sum:", row_sum.shape, "X_row_norm:", X_row_norm.shape)

X:
 [[ 0.91665479  0.37094684  0.61318908 -0.15219296]
 [-1.47388795  1.02885435 -1.93495964 -0.23993667]
 [-0.20452249 -1.04286014  0.61312314 -0.2003297 ]]
sigmoid(Y) shape: (3, 4)
sum over all: -1.7059213591939513
sum over rows (axis=1): (3,)
sum over cols (axis=0): (4,)
row_sum: (3, 1) X_row_norm: (3, 4)


## 6) Sorting, top-k, and `take_along_axis` (vectorized gather)

Most ranking/retrieval tasks boil down to:
- `argsort` / `argpartition` to get indices
- `take_along_axis` to gather aligned arrays (labels, doc_ids, etc.)

In [ ]:
scores = rng.normal(size=(3, 6))
rel = rng.random(size=(3, 6)) < 0.3  # random boolean relevance

order = np.argsort(-scores, axis=1)                 # (Q,N) sorted indices per row
rel_sorted = np.take_along_axis(rel, order, axis=1) # gather rel in sorted order

print("scores:\n", np.round(scores, 3))
print("order:\n", order)
print("rel:\n", rel.astype(int))
print("rel_sorted:\n", rel_sorted.astype(int))

# Top-k without fully sorting (faster for large N)
# k = 2
# topk_idx_unordered = np.argpartition(-scores, kth=k-1, axis=1)[:, :k]  # (Q,k), not sorted within top-k
# topk_scores = np.take_along_axis(scores, topk_idx_unordered, axis=1)
# # sort the top-k part only
# topk_order = np.argsort(-topk_scores, axis=1)
# topk_idx = np.take_along_axis(topk_idx_unordered, topk_order, axis=1)
# print("topk_idx:\n", topk_idx)

## 7) Linear algebra essentials

Know: `@` / `matmul`, `dot`, `einsum`, norms, SVD.

In [ ]:
A = rng.normal(size=(4, 3))
B = rng.normal(size=(3, 2))
C = A @ B  # (4,2)
print("A@B shape:", C.shape)

# dot vs matmul: dot is overloaded; matmul/@ is clearer for matrices
v = rng.normal(size=(3,))
print("A @ v shape:", (A @ v).shape)  # (4,)

# norms
print("||v||2:", np.linalg.norm(v))
print("row norms:", np.linalg.norm(A, axis=1))

# SVD: A = U S Vt
U, S, Vt = np.linalg.svd(A, full_matrices=False)
print("S shapes:", U.shape, S.shape, Vt.shape)

## 8) ML building blocks (vectorized)

These are interview “greatest hits”.

### 8.1 Standardization (z-score)

In [ ]:
X = rng.normal(size=(5, 3)) * 10 + 100  # (n,d)
mu = X.mean(axis=0, keepdims=True)
sigma = X.std(axis=0, keepdims=True) + 1e-12
Xz = (X - mu) / sigma
print("mean approx 0:", Xz.mean(axis=0))
print("std approx 1:", Xz.std(axis=0))

### 8.2 Stable softmax (row-wise)

In [ ]:
def softmax(Z):
    Z = np.asarray(Z)
    Zs = Z - np.max(Z, axis=1, keepdims=True)     # stability
    expZ = np.exp(Zs)
    return expZ / np.sum(expZ, axis=1, keepdims=True)

Z = rng.normal(size=(4, 5))
P = softmax(Z)
print("P row sums:", P.sum(axis=1))

### 8.3 Cosine similarity (batch)

X: (n,d), Y: (m,d) => sim: (n,m)

In [ ]:
X = rng.normal(size=(4, 3))
Y = rng.normal(size=(5, 3))
Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
Yn = Y / (np.linalg.norm(Y, axis=1, keepdims=True) + 1e-12)
sim = Xn @ Yn.T
print("sim shape:", sim.shape)
print(sim[:2, :3])

### 8.4 Pairwise squared Euclidean distances (broadcasting)

In [ ]:
X = rng.normal(size=(6, 2))
C = rng.normal(size=(4, 2))
d2 = np.sum((X[:, None, :] - C[None, :, :])**2, axis=2)  # (6,4)
print("d2 shape:", d2.shape)

### 8.5 K-means assignment + centroid update (one iteration)

This is the core of many “implement kmeans” interviews.

In [ ]:
def kmeans_one_iter(X, C):
    # X: (n,d), C: (k,d)
    d2 = np.sum((X[:, None, :] - C[None, :, :])**2, axis=2)  # (n,k)
    labels = np.argmin(d2, axis=1)                           # (n,)

    # centroid update (vectorized with bincount for each dim)
    k, d = C.shape
    counts = np.bincount(labels, minlength=k).astype(np.float64)  # (k,)
    # sum per cluster per dimension
    sums = np.vstack([np.bincount(labels, weights=X[:, j], minlength=k) for j in range(d)]).T  # (k,d)
    C_new = sums / (counts[:, None] + 1e-12)
    return labels, C_new

X = rng.normal(size=(20, 2))
C = X[rng.choice(X.shape[0], size=3, replace=False)]
labels, C2 = kmeans_one_iter(X, C)
print("labels shape:", labels.shape, "C2 shape:", C2.shape)

### 8.6 PCA via SVD (minimal)

In [ ]:
def pca_fit(X, n_components):
    X = np.asarray(X)
    mu = X.mean(axis=0, keepdims=True)
    Xc = X - mu
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    components = Vt[:n_components]  # (k,d)
    explained_var = (S**2) / (X.shape[0] - 1)
    explained_ratio = explained_var[:n_components] / explained_var.sum()
    return mu, components, explained_ratio

X = rng.normal(size=(200, 5))
X[:, 0] *= 5
mu, comps, ratio = pca_fit(X, n_components=2)
Z = (X - mu) @ comps.T
print("Z shape:", Z.shape, "explained ratio:", ratio)

### 8.7 MRR (Mean Reciprocal Rank) from scores + binary relevance

In [ ]:
def mrr_from_scores(scores, relevance):
    scores = np.asarray(scores)
    relevance = np.asarray(relevance).astype(bool)

    order = np.argsort(-scores, axis=1)  # (Q,N)
    rel_sorted = np.take_along_axis(relevance, order, axis=1)

    any_rel = rel_sorted.any(axis=1)          # (Q,)
    first_pos0 = np.argmax(rel_sorted, axis=1)  # (Q,) 0-based; returns 0 if none
    rr = any_rel.astype(float) / (first_pos0 + 1.0)
    return rr.mean()

scores = rng.normal(size=(4, 8))
rel = rng.random(size=(4, 8)) < 0.25
print("MRR:", mrr_from_scores(scores, rel))

## 9) Handy function cheat-sheet (things interviewers love)

**Creation:** `zeros`, `ones`, `full`, `eye`, `arange`, `linspace`, `meshgrid`

**Reshape/move axes:** `reshape`, `ravel`, `transpose`, `swapaxes`, `moveaxis`, `expand_dims`, `squeeze`

**Indexing helpers:** `where`, `take_along_axis`, `clip`, `argsort`, `argpartition`, `unique`, `bincount`

**Reductions:** `sum`, `mean`, `std`, `max`, `min`, `argmax`, `argmin`, `cumsum`, `cumprod`

**Linear algebra:** `@`, `einsum`, `linalg.norm`, `linalg.svd`, `linalg.solve`

**Random:** `default_rng().normal`, `.uniform`, `.integers`, `.choice`

In [ ]:
# A few quick “muscle memory” demos:
x = np.array([-2.0, -0.5, 0.2, 3.0])
print("clip:", np.clip(x, 0, 1))                # clamp
print("unique:", np.unique([1,2,2,3,1], return_counts=True))

# bincount: fast hist for non-negative ints
labels = np.array([0, 2, 2, 1, 0, 2])
print("bincount:", np.bincount(labels))

# meshgrid: useful for creating coordinate grids
xs = np.linspace(-1, 1, 3)
ys = np.linspace(-2, 2, 5)
Xg, Yg = np.meshgrid(xs, ys, indexing="xy")
print("grid shapes:", Xg.shape, Yg.shape)

## 10) Mini drills (5–10 minutes each)

These are deliberately short. Try to do them without googling; then verify by running.

1. Create a `(100, 10)` matrix `X` of standard normal values. Compute column means and standard deviations (with `keepdims=True`) and standardize it.

2. Given `scores: (Q,N)`, produce top-5 indices per query, sorted by score.

3. Given `X: (n,d)` and `C: (k,d)`, compute pairwise squared distances `(n,k)` without loops.

4. Implement stable softmax and check that each row sums to 1.

5. Given `y_true` and `y_pred` (0/1), compute precision, recall, F1 using vectorized ops.

In [ ]:
# (Optional) quick starting templates

# 1
X = rng.normal(size=(100, 10))
mu = X.mean(axis=0, keepdims=True)
sigma = X.std(axis=0, keepdims=True) + 1e-12
Xz = (X - mu) / sigma
print("means:", np.round(Xz.mean(axis=0), 3))
print("stds:", np.round(Xz.std(axis=0), 3))

# 2
Q, N, k = 4, 20, 5
scores = rng.normal(size=(Q, N))
topk = np.argpartition(-scores, kth=k-1, axis=1)[:, :k]
topk_scores = np.take_along_axis(scores, topk, axis=1)
topk_order = np.argsort(-topk_scores, axis=1)
topk_sorted = np.take_along_axis(topk, topk_order, axis=1)
print("topk_sorted shape:", topk_sorted.shape)

# 5
y_true = rng.integers(0, 2, size=50)
y_pred = rng.integers(0, 2, size=50)
tp = np.sum((y_true == 1) & (y_pred == 1))
fp = np.sum((y_true == 0) & (y_pred == 1))
fn = np.sum((y_true == 1) & (y_pred == 0))
prec = tp / (tp + fp + 1e-12)
rec  = tp / (tp + fn + 1e-12)
f1   = 2 * prec * rec / (prec + rec + 1e-12)
print("precision, recall, f1:", prec, rec, f1)

# Totally optional Advanced

In [ ]:
# 1) Scatter-add / accumulation with repeated indices: np.add.at (the safe one)
import numpy as np

idx  = np.array([0, 1, 1, 3, 3, 3])
vals = np.array([10, 1, 1, 5, 2, 2])

out_bad = np.zeros(5, dtype=int)
out_bad[idx] += vals  # ⚠️ may NOT accumulate correctly when idx has repeats (advanced indexing)

out_good = np.zeros(5, dtype=int)
np.add.at(out_good, idx, vals)  # ✅ guaranteed accumulation (true scatter-add)

print("out_bad :", out_bad)
print("out_good:", out_good)

# Bonus: vectorized "groupby sum" for cluster updates (like k-means), per-dimension
X = np.array([[1., 2.],
              [3., 4.],
              [5., 6.],
              [7., 8.]])             # (n=4, d=2)
labels = np.array([0, 1, 1, 0])       # (n,)
k = 2

counts = np.bincount(labels, minlength=k)                 # (k,)
sums = np.vstack([np.bincount(labels, X[:, j], minlength=k) for j in range(X.shape[1])]).T  # (k,d)
centroids = sums / (counts[:, None] + 1e-12)
print("centroids:\n", centroids)


In [ ]:
# 2) Sliding windows + NaN-safe stats + einsum + stable logsumexp (all in one “interview” cell)
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

# Sliding window (rolling features) without loops
x = np.arange(10)
w = 4
win = sliding_window_view(x, window_shape=w)   # shape (10-w+1, w)
mov_avg = win.mean(axis=1)
print("windows shape:", win.shape)
print("moving avg:", mov_avg)

# NaN-aware reductions
a = np.array([1.0, np.nan, 3.0, np.nan, 5.0])
print("mean:", np.mean(a))          # nan
print("nanmean:", np.nanmean(a))    # 3.0

# einsum: pairwise dot products + row-wise squared norms
rng = np.random.default_rng(0)
X = rng.normal(size=(4, 3))  # (n,d)
Y = rng.normal(size=(5, 3))  # (m,d)

dots = np.einsum("nd,md->nm", X, Y)     # (n,m) pairwise dot products
nrm2 = np.einsum("nd,nd->n",  X, X)     # (n,)  squared L2 norms
print("dots shape:", dots.shape)
print("nrm2:", nrm2)

# Stable logsumexp (softmax's numerically-stable cousin)
def logsumexp(Z, axis=1):
    Z = np.asarray(Z)
    m = np.max(Z, axis=axis, keepdims=True)
    return (m + np.log(np.sum(np.exp(Z - m), axis=axis, keepdims=True))).squeeze(axis=axis)

Z = np.array([[1000.0, 1001.0,  999.0],
              [   0.0,   -1.0,   -2.0]])
print("logsumexp:", logsumexp(Z, axis=1))

# Bonus: "solve > inv" (numerical hygiene)
A = rng.normal(size=(4, 4))
b = rng.normal(size=(4,))
x_sol = np.linalg.solve(A, b)          # preferred
print("residual max|Ax-b|:", np.max(np.abs(A @ x_sol - b)))


In [26]:
rng = np.random.default_rng()

In [27]:
rng.normal(size = [4,5])

array([[ 0.38526352, -1.76534121,  1.30196969,  0.77177468,  1.59899699],
       [ 1.13375087,  1.85763803,  0.4384149 , -1.71639408, -0.10006733],
       [ 1.22982038, -0.28806388,  1.43532486,  1.16799776, -0.5515116 ],
       [ 0.35211448,  0.5427394 ,  0.82306521, -0.8900789 ,  0.3042607 ]])

In [29]:
1e-12 + 1e-12

2e-12

In [30]:
a = np.array([1,2])

In [37]:
b = np.append(a, np.array([0]))

In [39]:
np.shares_memory(a,a)

True